# Lesson 5B: N-Step Methods & Eligibility Traces - Practical

## Introduction

In Lesson 5A, we learned TD(λ) and eligibility traces—the mathematical framework for n-step returns.

Now we implement **Sarsa(λ)** on **RandomWalk**—the canonical environment for seeing how λ trades off bias and variance.

### RandomWalk Problem

- 1D chain: states 0-6 (left to right)
- Start at state 3
- Actions: left (-1) or right (+1)
- Reward: 0 except terminal (−1 at left, +1 at right)
- True values: V(0)=0, V(1)=1/6, V(2)=2/6, ..., V(6)=1

Perfect for analyzing prediction error because ground truth is known analytically.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
print('N-Step Eligibility Traces Practical')

## RandomWalk Environment

In [ ]:
class RandomWalk:
    def __init__(self, n_states=7):
        self.n_states = n_states
        self.start_state = n_states // 2
        self.state = self.start_state
        self.true_values = np.linspace(0, 1, n_states)  # True V(s)
    
    def reset(self):
        self.state = self.start_state
        return self.state
    
    def step(self, action):
        # action: 0=left, 1=right
        if action == 0:
            self.state -= 1
        else:
            self.state += 1
        
        done = self.state < 0 or self.state >= self.n_states
        reward = 0
        
        if self.state < 0:
            reward = -1
        elif self.state >= self.n_states:
            reward = 1
        
        return self.state if not done else None, reward, done

env = RandomWalk()
print(f'True values: {env.true_values}')

## Sarsa(λ) Implementation

In [ ]:
def sarsa_lambda(env, lam=0.5, alpha=0.1, gamma=0.99, episodes=100):
    """
    Sarsa(λ) on RandomWalk.
    """
    V = np.zeros(env.n_states)
    e = np.zeros(env.n_states)  # Eligibility traces
    errors = []
    
    for ep in range(episodes):
        s = env.reset()
        e.fill(0)  # Reset traces
        episode_error = 0
        step_count = 0
        done = False
        
        while not done and step_count < 1000:
            # Take random action
            a = np.random.randint(2)
            s_next, r, done = env.step(a)
            step_count += 1
            
            # TD error
            v_next = V[s_next] if s_next is not None else 0
            delta = r + gamma * v_next - V[s]
            
            # Update traces
            e[s] += 1  # Accumulating trace
            
            # Update all states
            for i in range(env.n_states):
                V[i] += alpha * delta * e[i]
                e[i] *= gamma * lam  # Decay traces
            
            episode_error += np.abs(delta)
            s = s_next if s_next is not None else -1
        
        errors.append(episode_error)
    
    return V, np.array(errors)

# Test with different lambda values
print('Testing Sarsa(λ) with different lambda values:')
lambdas = [0.0, 0.3, 0.7, 1.0]
results = {}

for lam in lambdas:
    V_learned, errors = sarsa_lambda(env, lam=lam, episodes=200)
    mse = np.mean((V_learned - env.true_values) ** 2)
    results[lam] = (V_learned, errors, mse)
    print(f'λ={lam}: MSE={mse:.4f}')

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves
ax = axes[0]
for lam, (_, errors, _) in results.items():
    window = 20
    smoothed = np.convolve(errors, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'λ={lam}', linewidth=2)

ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('TD Error (Smoothed)', fontsize=11)
ax.set_title('Sarsa(λ) Learning Curves', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Value function comparison
ax = axes[1]
ax.plot(env.true_values, 'ko-', label='True V(s)', linewidth=3, markersize=8)

for lam, (V_learned, _, _) in results.items():
    ax.plot(V_learned, 'o-', label=f'Learned λ={lam}', alpha=0.7, linewidth=2)

ax.set_xlabel('State', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Learned vs True Values', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Findings

1. **λ=0 (TD(0))**: High bias (bootstraps too much), fast convergence, underestimates
2. **λ=1 (MC)**: High variance (full trajectory), slow convergence, unbiased
3. **λ∈(0,1)**: Sweet spot—balanced bias and variance
4. **Eligibility traces**: Credit assignment efficiently—all visited states updated immediately

### Why Eligibility Traces Matter

- Update all states every step (not waiting n steps)
- Recency weighting: recent states get more credit
- Bridges DP (fast) and MC (unbiased) with single parameter λ

### Next: Lesson 6

**Function Approximation**—scale to large/continuous state spaces using neural networks.